# 04 — Inferenza dei Layer Normativi e Heatmap di Ibridità

Questo notebook implementa il cuore metodologico del progetto: **i livelli gerarchici
non sono predefiniti ma emergono dai dati** della specifica materia analizzata.

## Pipeline in quattro fasi

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Segmenti di ogni atto | LLM 1: descrizione funzionale contestualizzata per segmento | `segments_descriptions.csv` |
| **B** | Descrizioni funzionali | Embedding + UMAP + HDBSCAN → cluster = layer emergenti | — |
| **B.2** | 10 descrizioni representative per cluster | LLM 2: nome del layer | `layer_mapping.csv` |
| **C** | Articoli + layer noti | LLM 3: distribuzione % articolo × layer | `nodes_heatmap.csv` |
| **D** | Matrice per atto | Entropia media → score di ibridità continuo | `nodes_hybridity.csv` |

## Principio metodologico

Il clustering avviene a livello di **segmento** (articolo o considerando), non di atto.
I cluster che emergono rappresentano i livelli gerarchici specifici per quella materia —
quanti siano lo decide l'algoritmo, non l'analista.

Un atto è **ibrido** se i suoi articoli hanno distribuzioni molto diverse tra loro:
alcuni concentrati su livelli apicali, altri su livelli tecnici di dettaglio.
L'ibridità è misurata come **entropia media degli articoli**.

## Output

| File | Contenuto |
|---|---|
| `segments_descriptions.csv` | Una riga per segmento con la descrizione funzionale (LLM 1) |
| `layer_mapping.csv` | Cluster → nome layer con descrizione e rank gerarchico |
| `nodes_heatmap.csv` | Una riga per (celex, articolo) con % per ogni layer |
| `nodes_hybridity.csv` | Una riga per atto con score ibridità e layer dominante |

---

> **Nota sul costo API**: Fase A chiama l'LLM una volta per segmento, Fase C una volta
> per articolo. Con ~200 atti e ~20 segmenti/atto = ordine di 4.000–6.000 chiamate.
> Il checkpointing granulare permette di riprendere da dove si era interrotti.

## 0. Configurazione

**Modifica solo questa cella.** Il resto del notebook gira in automatico.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  MATERIA  →  stessa cartella scelta nei notebook 02 e 03
# ─────────────────────────────────────────────────────────────────────────────

MATERIA_NAME = "fdi_screening"   # <- modifica qui


TEMA_DESCRIZIONE = (
    "Foreign direct investment screening (FDI Screening)"
    "and special state powers (Golden Power) over national security"
    "and strategic infrastructure."
)


# ── Modello OpenAI ─────────────────────────────────────────────────────────────
LLM_MODEL = "gpt-5.4-mini"   


# ── Parametri chiamate API ────────────────────────────────────────────────────
LLM_MAX_TOKENS_A   = 300    # Fase A: descrizione livello di astrazione (2-3 frasi)
LLM_MAX_TOKENS_A2  = 200    # Fase A2: JSON distribuzione % Lamfalussy
LLM_MAX_TOKENS_B2  = 400    # Fase B.2: JSON nome + descrizione layer
LLM_MAX_TOKENS_C   = 500    # Fase C: JSON percentuali
LLM_DELAY_SECONDS  = 0.3    # pausa tra chiamate (rispetta il rate limit)
LLM_MAX_RETRIES    = 3      # tentativi in caso di errore transitorio
LLM_RETRY_DELAY    = 5.0    # secondi tra retry


# ── Parametri checkpoint ──────────────────────────────────────────────────────
CHECKPOINT_EVERY   = 100    # segmenti/articoli tra un salvataggio e il successivo


# ── Parametri clustering ──────────────────────────────────────────────────────
EMBEDDING_MODEL     = "all-mpnet-base-v2"
UMAP_N_COMPONENTS   = 10     # dimensioni ridotte prima di HDBSCAN
UMAP_N_NEIGHBORS    = 15
UMAP_MIN_DIST       = 0.0    # 0.0 ottimizza la separazione dei cluster

HDBSCAN_MIN_CLUSTER = 50
HDBSCAN_MIN_SAMPLES = 10
N_REPR_DESCRIPTIONS = 10     # descrizioni representative per il naming

## 1. Import e Percorsi

In [2]:
import os
import json
import math
import time
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path  = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

INPUT_FILE              = os.path.join(output_path, 'nodes_texts.csv')
EDGES_FILE              = os.path.join(output_path, 'edges_focal.csv')
SEGMENTS_DESC_FILE      = os.path.join(output_path, 'segments_descriptions.csv')
LAYER_MAPPING_FILE      = os.path.join(output_path, 'layer_mapping.csv')
NODES_HEATMAP_FILE      = os.path.join(output_path, 'nodes_heatmap.csv')
NODES_HYBRIDITY_FILE    = os.path.join(output_path, 'nodes_hybridity.csv')
HEATMAP_CKPT_FILE       = os.path.join(output_path, 'heatmap_checkpoint.csv')

SEGMENTS_LAMF_FILE      = os.path.join(output_path, 'segments_lamfalussy.csv')
SEGMENTS_LAMF_CKPT_FILE = os.path.join(output_path, 'segments_lamfalussy_checkpoint.csv')
NODES_LAMFALUSSY_FILE   = os.path.join(output_path, 'nodes_lamfalussy.csv')  # articoli filtrati, per viz

print(f"Materia:       {MATERIA_NAME}")
print(f"Input:         {INPUT_FILE}")
print(f"Modello LLM:   {LLM_MODEL}")
print(f"Embedding:     {EMBEDDING_MODEL}")

Materia:       fdi_screening
Input:         ..\data\output\fdi_screening\nodes_texts.csv
Modello LLM:   gpt-5.4-mini
Embedding:     all-mpnet-base-v2


## 2. Caricamento Dati

In [3]:
nodes = pd.read_csv(INPUT_FILE)
print(f"Nodi totali: {len(nodes)}")

REQUIRED_COLS = ['Id', 'Label', 'segments', 'text_status', 'title']
missing_cols = [c for c in REQUIRED_COLS if c not in nodes.columns]
if missing_cols:
    raise RuntimeError(
        f"Colonne mancanti in {INPUT_FILE}: {missing_cols}.\n"
        "Assicurarsi che il notebook 03 sia stato eseguito completamente."
    )

nodes_ok   = nodes[nodes['text_status'] == 'ok'].copy()
nodes_fail = nodes[nodes['text_status'] != 'ok'].copy()

print(f"Atti con testo (text_status=ok): {len(nodes_ok)}")
print(f"Atti senza testo (esclusi):      {len(nodes_fail)}")
print()
print("Distribuzione text_status:")
print(nodes['text_status'].value_counts().to_string())

Nodi totali: 19
Atti con testo (text_status=ok): 19
Atti senza testo (esclusi):      0

Distribuzione text_status:
text_status
ok    19


## 3. Divisione in Segmenti

La colonna `segments` di ogni atto contiene una lista JSON di segmenti strutturati
(articoli, considerando, allegati). Questa cella costruisce un DataFrame flat
`segments_df` con **una riga per segmento** — l'unità di analisi del clustering.

In [4]:
def parse_segments(row):
    """Parsa la colonna 'segments' e restituisce lista di dict arricchiti."""
    celex = row.get('Label', row['Id'])
    title = str(row.get('title', ''))
    raw   = row.get('segments')

    if pd.isna(raw) or not str(raw).strip():
        return []
    try:
        segs = json.loads(str(raw))
    except (json.JSONDecodeError, ValueError):
        return []

    result = []
    for s in segs:
        result.append({
            'celex':          celex,
            'node_id':        row['Id'],
            'title_atto':     title,
            'tipo':           s.get('tipo', ''),
            'identificatore': s.get('identificatore', ''),
            'testo':          s.get('testo', ''),
        })
    return result


all_segments = []
for _, row in nodes_ok.iterrows():
    all_segments.extend(parse_segments(row))

segments_df = pd.DataFrame(all_segments)

# ID univoco per segmento
segments_df['segment_id'] = (
    segments_df['celex'] + '__' +
    segments_df['tipo'] + '__' +
    segments_df['identificatore'].astype(str)
)

# Rimuove duplicati su segment_id (stesso atto, stesso tipo, stesso identificatore)
before_dedup = len(segments_df)
segments_df = segments_df.drop_duplicates(subset='segment_id', keep='first').reset_index(drop=True)

# Rimuove segmenti con testo troppo breve per essere informativi
MIN_TESTO_LEN = 30
before = len(segments_df)
segments_df = segments_df[segments_df['testo'].str.len() >= MIN_TESTO_LEN].reset_index(drop=True)

# Esclude considerando e header preambolo — non entrano nel clustering né nella heatmap
TIPI_ESCLUSI = {'considerando', 'preambolo_header'}
before_tipi = len(segments_df)
segments_df = segments_df[~segments_df['tipo'].isin(TIPI_ESCLUSI)].reset_index(drop=True)

print(f"Segmenti estratti:              {before_dedup:,}")
print(f"Segmenti scartati (duplicati):  {before_dedup - before:,}")
print(f"Segmenti validi (>={MIN_TESTO_LEN} car): {len(segments_df):,}")
print(f"Segmenti scartati (testo):      {before - before_tipi:,}")
print(f"Segmenti scartati (tipo):       {before_tipi - len(segments_df):,}")
print()
print("Distribuzione per tipo:")
print(segments_df['tipo'].value_counts().to_string())
print()
print(f"Segmenti medi per atto:  {segments_df.groupby('celex').size().mean():.1f}")

Segmenti estratti:              1,249
Segmenti scartati (duplicati):  53
Segmenti validi (>=30 car): 848
Segmenti scartati (testo):      7
Segmenti scartati (tipo):       341

Distribuzione per tipo:
tipo
articolo    848

Segmenti medi per atto:  44.6


## 4. Fase A — Livello di Astrazione per Segmento (LLM 1)

Per ogni segmento l'LLM produce una **descrizione libera del livello di astrazione**:
dove si colloca il segmento nella piramide normativa — quanto è fondazionale vs tecnico/operativo.

La descrizione è volutamente domain-agnostic e non usa la terminologia Lamfalussy:
non vengono imposti tag o categorie. I livelli specifici per materia emergeranno
liberamente dal clustering in Fase B.

**Fase A2** (successiva) assegnerà poi i livelli Lamfalussy standard (L1–L4)
usando queste descrizioni come contesto.

> **Checkpoint**: i risultati vengono salvati in `segments_descriptions.csv` ogni
> `CHECKPOINT_EVERY` segmenti. Rieseguire la cella riprende dal punto di interruzione.

In [5]:
def build_prompt_functional_description(testo, tipo, identificatore, title_atto, tema):
    return f"""You are an expert in European law.

Your task is to describe the position of this legal segment in the regulatory
hierarchy — how general or specific it is, and why.

Focus exclusively on the ABSTRACTION LEVEL:
Does this segment state a broad principle that governs the entire framework,
or does it implement a narrow technical detail that only applies in a specific
situation? Where on the spectrum from foundational to operational does it sit?

Describe:
1. Where it sits on the spectrum (foundational / structural / operational / technical)
2. Why — what feature of the text places it there (e.g., it establishes a purpose,
   it delegates a power, it specifies a procedure, it fixes a threshold,
   it sets an effective date, it defines a concept)

## Critical rules
- Do NOT describe what the segment is about thematically.
- Do NOT mention the specific subject matter (FDI, data protection, banking,
  subsidies, etc.).
- Describe only the position in the normative hierarchy and its structural reason.
- 2-3 sentences maximum.

## Examples of correct descriptions
- "Foundational recital that states the overarching policy rationale justifying
   the entire legislative intervention — sits at the highest level of abstraction
   as it frames the purpose of the whole framework without specifying any operative rule."
- "Structural empowerment clause delegating secondary rule-making power to an
   institution within defined substantive limits — sits at an intermediate level,
   organising institutional competences without specifying how they must be exercised."
- "Narrow operative criterion specifying one concrete factor to be checked during
   a particular assessment — highly specific, functioning as a technical sub-rule
   within a broader procedure."
- "Terminal commencement clause fixing the date the act becomes legally effective —
   sits at the most technical end, containing no substantive normative content."

## Also avoid
- Descriptions so vague they say nothing: "Sets out a provision within the framework."
- Restating what the text says without identifying the hierarchical position.
- Any reference to the subject matter of the regulation.

Act title: {title_atto}
Segment ({tipo} {identificatore}):
{testo}

Reply ONLY with the description (2-3 sentences), in English, no other text."""

def call_llm(client, prompt, max_tokens):
    """
    Chiama l'LLM OpenAI con retry automatico su errori transitori.
    Restituisce (testo_risposta, status) dove status è 'ok' | 'error' | 'empty'.
    """
    for attempt in range(LLM_MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                max_completion_tokens=max_tokens,
                temperature=0.3,
                messages=[{'role': 'user', 'content': prompt}]
            )
            text = response.choices[0].message.content.strip()
            if not text:
                return '', 'empty'
            return text, 'ok'

        except openai.RateLimitError:
            wait = LLM_RETRY_DELAY * (attempt + 1) * 2
            print(f"  [RateLimit] attesa {wait:.0f}s (tentativo {attempt+1}/{LLM_MAX_RETRIES})")
            time.sleep(wait)

        except openai.APIStatusError as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

        except Exception as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

    return 'ERROR: max retries exceeded', 'error'


print("Funzioni LLM definite.")

Funzioni LLM definite.


In [6]:
# ── TEST su 5 segmenti ────────────────────────────────────────────────────────
'''from openai import OpenAI

client = OpenAI()


test_segments = segments_df.sample(5, random_state=42)

for _, seg in test_segments.iterrows():
    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)
    
    print(f"CELEX: {seg['celex']}  |  {seg['tipo']} {seg['identificatore']}")
    print(f"Testo: {seg['testo'][:150]}...")
    print(f"→ {descrizione}")
    print()'''

'from openai import OpenAI\n\nclient = OpenAI()\n\n\ntest_segments = segments_df.sample(5, random_state=42)\n\nfor _, seg in test_segments.iterrows():\n    prompt = build_prompt_functional_description(\n        testo          = seg[\'testo\'],\n        tipo           = seg[\'tipo\'],\n        identificatore = seg[\'identificatore\'],\n        title_atto     = seg[\'title_atto\'],\n        tema           = TEMA_DESCRIZIONE,\n    )\n    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)\n\n    print(f"CELEX: {seg[\'celex\']}  |  {seg[\'tipo\']} {seg[\'identificatore\']}")\n    print(f"Testo: {seg[\'testo\'][:150]}...")\n    print(f"→ {descrizione}")\n    print()'

In [7]:
# ── Gestione checkpoint ───────────────────────────────────────────────────────
if os.path.exists(SEGMENTS_DESC_FILE):
    segs_done = pd.read_csv(SEGMENTS_DESC_FILE)
    done_ids  = set(segs_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_ids):,} segmenti già descritti.")
else:
    segs_done = pd.DataFrame()
    done_ids  = set()
    print("Nessun checkpoint — si parte da zero.")

segments_todo = segments_df[~segments_df['segment_id'].isin(done_ids)].copy()
print(f"Da descrivere: {len(segments_todo):,}")

if len(segments_todo) == 0:
    print("✓ Tutti i segmenti già descritti — si può passare alla Fase B.")

Checkpoint trovato: 848 segmenti già descritti.
Da descrivere: 0
✓ Tutti i segmenti già descritti — si può passare alla Fase B.


In [8]:
%%time
from openai import OpenAI

client = OpenAI()

new_rows = []
n_ok = n_error = 0
total = len(segments_todo)

for i, (_, seg) in enumerate(segments_todo.iterrows()):

    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)

    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)
    if status != 'ok':
        print(f"ERRORE: {descrizione}")
        break   # blocca subito al primo errore per leggere il messaggio

    new_rows.append({
        'segment_id':              seg['segment_id'],
        'celex':                   seg['celex'],
        'node_id':                 seg['node_id'],
        'tipo':                    seg['tipo'],
        'identificatore':          seg['identificatore'],
        'testo_originale':         seg['testo'],
        'descrizione_funzionale':  descrizione,
        'llm_status':              status,
    })

    if status == 'ok':
        n_ok += 1
    else:
        n_error += 1

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([segs_done, batch], ignore_index=True) if not segs_done.empty else batch
        combined.to_csv(SEGMENTS_DESC_FILE, index=False)
        print(f"  [{i+1:>5}/{total}]  {(i+1)/total*100:5.1f}%   ok: {n_ok}   errori: {n_error}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE A — ok: {n_ok:,}   errori: {n_error:,}")
print("=" * 50)


FASE A — ok: 0   errori: 0
CPU times: total: 250 ms
Wall time: 402 ms


## 4b. Fase A2 — Assegnazione Livelli Lamfalussy per Segmento (LLM 2)

Partendo dall'output di Fase A (`segments_descriptions.csv`), per ogni segmento
l'LLM assegna una distribuzione percentuale sui 4 livelli Lamfalussy.

La **descrizione del livello di astrazione** prodotta in Fase A viene fornita come
contesto: l'LLM non deve più dedurre la posizione gerarchica dal testo,
deve solo mappare quella descrizione sulla tassonomia Lamfalussy standard.

**Output**: `segments_lamfalussy.csv` — una riga per segmento con
`lamf_L1`, `lamf_L2`, `lamf_L3`, `lamf_L4` (somma = 100).

> Gira su **tutti** i segmenti (considerando + articoli + allegati).
> Fase C2 filtrerà ai soli articoli per il calcolo dell'entropia.

In [9]:
# ── Costanti Lamfalussy (usate in A2, C2, visualizzazioni) ────────────────────
LAMFALUSSY_LEVELS = [
    {'key': 'L1', 'name': 'L1 — Framework principles',
     'description': 'Constitutive norms establishing purpose, scope, legal basis '
                    'and fundamental objectives of the regulatory framework.'},
    {'key': 'L2', 'name': 'L2 — Implementing measures',
     'description': 'Operational rules translating L1 principles — empowerment clauses, '
                    'delegated powers, detailed procedures, definitions, thresholds.'},
    {'key': 'L3', 'name': 'L3 — Coordination and convergence',
     'description': 'Norms harmonising application across jurisdictions — cooperation '
                    'mechanisms, information sharing, supervisory convergence, technical standards.'},
    {'key': 'L4', 'name': 'L4 — Enforcement and monitoring',
     'description': 'Norms ensuring compliance — reporting obligations, monitoring, '
                    'sanctions, review clauses, entry-into-force and transitional provisions.'},
]

LAMF_KEYS = [l['key'] for l in LAMFALUSSY_LEVELS]
LAMF_COLS = [f'lamf_{k}' for k in LAMF_KEYS]   # ['lamf_L1','lamf_L2','lamf_L3','lamf_L4']


def build_prompt_lamfalussy_assignment(testo, identificatore, tipo, descrizione_astrazione):
    levels_desc = '\n'.join([
        f"  {l['name']}: {l['description']}"
        for l in LAMFALUSSY_LEVELS
    ])
    return f"""You are an expert in European law.

A colleague has already described the hierarchical position of the segment below:

  Hierarchy description: {descrizione_astrazione}

Using that description as guidance, distribute the content of this segment
as a percentage across the four Lamfalussy levels.

Lamfalussy levels:
{levels_desc}

Rules:
- Percentages must sum to exactly 100.
- Assign 0 to levels not present in the segment.
- If the segment genuinely spans multiple levels, reflect the actual proportions.
- Let the hierarchy description guide your assignment — do not re-read the text
  from a thematic perspective.

Segment ({tipo} {identificatore}):
{testo}

Reply ONLY with valid JSON (no other text, no backticks):
{{"L1": X, "L2": X, "L3": X, "L4": X}}"""


print("Costanti Lamfalussy e funzioni Fase A2 definite.")
print(f"Livelli: {LAMF_KEYS}")


Costanti Lamfalussy e funzioni Fase A2 definite.
Livelli: ['L1', 'L2', 'L3', 'L4']


In [10]:
%%time
# ── Gestione checkpoint ────────────────────────────────────────────────────────
if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
    lamf_segs_done = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    done_lamf_ids  = set(lamf_segs_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_lamf_ids):,} segmenti già classificati.")
else:
    lamf_segs_done = pd.DataFrame()
    done_lamf_ids  = set()
    print("Nessun checkpoint A2 — si parte da zero.")

# Input: output Fase A (contiene testo + descrizione astrazione)
segs_a_out = pd.read_csv(SEGMENTS_DESC_FILE)
segs_a_ok  = segs_a_out[segs_a_out['llm_status'] == 'ok'].copy()
segs_a_todo = segs_a_ok[~segs_a_ok['segment_id'].isin(done_lamf_ids)].copy()
print(f"Segmenti da classificare: {len(segs_a_todo):,}  (totale ok: {len(segs_a_ok):,})")
print()

new_lamf_rows = []
n_ok_a2 = n_error_a2 = 0
total_a2 = len(segs_a_todo)

for i, (_, seg) in enumerate(segs_a_todo.iterrows()):

    prompt = build_prompt_lamfalussy_assignment(
        testo                  = seg['testo_originale'],
        identificatore         = seg['identificatore'],
        tipo                   = seg['tipo'],
        descrizione_astrazione = seg['descrizione_funzionale'],
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_A2)

    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, LAMF_KEYS)
        if distribution is None:
            status = 'parse_error'

    row = {
        'segment_id':    seg['segment_id'],
        'celex':         seg['celex'],
        'node_id':       seg['node_id'],
        'tipo':          seg['tipo'],
        'identificatore': seg['identificatore'],
        'llm_status':    status,
    }
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        row[col] = round(distribution[key], 2) if distribution else 0.0

    if distribution:
        n_ok_a2 += 1
    else:
        n_error_a2 += 1

    new_lamf_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total_a2:
        batch    = pd.DataFrame(new_lamf_rows)
        combined = pd.concat([lamf_segs_done, batch], ignore_index=True) if not lamf_segs_done.empty else batch
        combined.to_csv(SEGMENTS_LAMF_CKPT_FILE, index=False)
        print(f"  [{i+1:>5}/{total_a2}]  {(i+1)/total_a2*100:5.1f}%   ok: {n_ok_a2}   errori: {n_error_a2}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print('=' * 50)
print(f"FASE A2 — ok: {n_ok_a2:,}   errori: {n_error_a2:,}")
print('=' * 50)


Checkpoint trovato: 848 segmenti già classificati.
Segmenti da classificare: 0  (totale ok: 848)


FASE A2 — ok: 0   errori: 0
CPU times: total: 0 ns
Wall time: 27.1 ms


In [11]:
# Salva output Fase A2
segments_lamf_final = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
segments_lamf_final.to_csv(SEGMENTS_LAMF_FILE, index=False)

print(f"Salvato: {SEGMENTS_LAMF_FILE}")
print(f"Segmenti totali: {len(segments_lamf_final):,}  |  Colonne Lamfalussy: {LAMF_COLS}")
print()

ok_mask = segments_lamf_final['llm_status'] == 'ok'
print("Distribuzione media % per livello Lamfalussy (tutti i segmenti ok):")
for col, key in zip(LAMF_COLS, LAMF_KEYS):
    level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
    mean_pct = segments_lamf_final.loc[ok_mask, col].mean()
    bar      = '█' * int(mean_pct / 2)
    print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")

print()
# Breakdown per tipo di segmento
print("Distribuzione per tipo di segmento:")
for tipo in ['considerando', 'articolo', 'allegato']:
    n = ok_mask & (segments_lamf_final['tipo'] == tipo)
    if n.sum() > 0:
        print(f"  {tipo:>12}: {n.sum():>4} segmenti")


Salvato: ..\data\output\fdi_screening\segments_lamfalussy.csv
Segmenti totali: 848  |  Colonne Lamfalussy: ['lamf_L1', 'lamf_L2', 'lamf_L3', 'lamf_L4']

Distribuzione media % per livello Lamfalussy (tutti i segmenti ok):
  L1 — Framework principles.....................   2.9%  █
  L2 — Implementing measures....................  70.7%  ███████████████████████████████████
  L3 — Coordination and convergence.............   4.8%  ██
  L4 — Enforcement and monitoring...............  21.6%  ██████████

Distribuzione per tipo di segmento:
      articolo:  848 segmenti


## 5. Fase B — Embedding e Clustering → Layer Emergenti

Le descrizioni funzionali vengono embeddate con `all-mpnet-base-v2`, ridotte con
UMAP e clusterizzate con HDBSCAN. Ogni cluster che emerge è un **livello gerarchico
specifico per questa materia** — quanti ce ne sono lo decide l'algoritmo.

I segmenti assegnati al cluster `-1` (noise) vengono conservati ma esclusi dal naming.

In [12]:
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# Carica dal checkpoint finale
segs_desc = pd.read_csv(SEGMENTS_DESC_FILE)

segs_valid = segs_desc[
    (segs_desc['llm_status'] == 'ok') &
    segs_desc['descrizione_funzionale'].notna() &
    (segs_desc['descrizione_funzionale'].str.len() > 10)
].copy()

print(f"Segmenti con descrizione valida: {len(segs_valid):,} / {len(segs_desc):,}")
print()

# ── Embedding con cache ───────────────────────────────────────────────────────
embeddings_file = os.path.join(output_path, 'embeddings.npy')

if os.path.exists(embeddings_file):
    embeddings = np.load(embeddings_file)
    print(f"Embeddings caricati da cache: {embeddings.shape}")
else:
    print(f"Caricamento modello embedding: {EMBEDDING_MODEL} ...")
    encoder = SentenceTransformer(EMBEDDING_MODEL)
    print("Calcolo embeddings...")
    embeddings = encoder.encode(
        segs_valid['descrizione_funzionale'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print(f"Embeddings shape: {embeddings.shape}")
    np.save(embeddings_file, embeddings)
    print(f"Embeddings salvati: {embeddings_file}")

Segmenti con descrizione valida: 848 / 848

Embeddings caricati da cache: (848, 768)


In [13]:
import numpy as np

embeddings_file = os.path.join(output_path, 'embeddings.npy')
np.save(embeddings_file, embeddings)
print(f"Embeddings salvati: {embeddings_file}")

Embeddings salvati: ..\data\output\fdi_screening\embeddings.npy


In [14]:
print(f"UMAP: {embeddings.shape[1]}d → {UMAP_N_COMPONENTS}d ...")

reducer = umap.UMAP(
    n_components = UMAP_N_COMPONENTS,
    n_neighbors  = UMAP_N_NEIGHBORS,
    min_dist     = UMAP_MIN_DIST,
    metric       = 'cosine',
    random_state = 42,
    low_memory   = False,
)
embeddings_reduced = reducer.fit_transform(embeddings)
print(f"Shape ridotta: {embeddings_reduced.shape}")

UMAP: 768d → 10d ...


c:\Users\claud\Documents\GitHub\eu-law-network-viz\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Shape ridotta: (848, 10)


In [15]:
print(f"HDBSCAN (min_cluster={HDBSCAN_MIN_CLUSTER}, min_samples={HDBSCAN_MIN_SAMPLES}) ...")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size         = HDBSCAN_MIN_CLUSTER,
    min_samples              = HDBSCAN_MIN_SAMPLES,
    cluster_selection_method = 'eom',
    prediction_data          = True,
)
cluster_labels = clusterer.fit_predict(embeddings_reduced)

segs_valid = segs_valid.copy()
segs_valid['cluster_id'] = cluster_labels
if hasattr(clusterer, 'probabilities_'):
    segs_valid['cluster_prob'] = clusterer.probabilities_

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()

print()
print("=" * 50)
print("RISULTATO CLUSTERING")
print("=" * 50)
print(f"  Layer trovati:          {n_clusters}")
print(f"  Segmenti noise (-1):    {n_noise:,}  ({n_noise/len(cluster_labels)*100:.1f}%)")
print()

cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print("Distribuzione segmenti per cluster:")
for cid, cnt in cluster_counts.items():
    tag = 'NOISE' if cid == -1 else f'cluster_{cid}'
    bar = '█' * min(40, int(cnt / cluster_counts.max() * 40))
    print(f"  {tag:>12}: {cnt:>5}  {bar}")

HDBSCAN (min_cluster=50, min_samples=10) ...

RISULTATO CLUSTERING
  Layer trovati:          5
  Segmenti noise (-1):    126  (14.9%)

Distribuzione segmenti per cluster:
         NOISE:   126  █████████████████
     cluster_0:   137  ███████████████████
     cluster_1:   282  ████████████████████████████████████████
     cluster_2:    95  █████████████
     cluster_3:   108  ███████████████
     cluster_4:   100  ██████████████


## 6. Fase B.2 — Ordinamento Gerarchico e Naming dei Layer (LLM 2)

**Naming**: LLM 2 riceve le 10 descrizioni più rappresentative di ogni cluster e assegna un nome e una descrizione al layer,
ed infine li ordina gerarchicamente.

In [16]:
from openai import OpenAI

client = OpenAI()
   

# ── Funzioni ──────────────────────────────────────────────────────────────────

def call_llm(client, prompt, max_tokens):
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=max_tokens,
        )
        return response.choices[0].message.content.strip(), 'ok'
    except Exception as e:
        print(f"  Errore LLM: {e}")
        return '', 'error'


def get_representative_descriptions(segs_df, cluster_id, n=N_REPR_DESCRIPTIONS):
    subset = segs_df[segs_df['cluster_id'] == cluster_id].copy()
    if 'cluster_prob' in subset.columns:
        subset = subset.nlargest(n, 'cluster_prob')
    else:
        subset = subset.sample(min(n, len(subset)), random_state=42)
    return subset['descrizione_funzionale'].tolist()


def build_prompt_layer_naming(descriptions):
    desc_list = '\n'.join([f"  {i+1}. {d}" for i, d in enumerate(descriptions)])
    return f"""You are an expert in European law.

Below are functional descriptions of legal segments that cluster together 
in a corpus analysis. They share the same normative role in the regulatory 
hierarchy.

Identify what hierarchical function unites them and give this cluster:
- a SHORT name (3-6 words, functional not thematic)
- a brief description (2-3 sentences) of the normative role

GOOD name examples:
- "Empowerment and delegation clauses"
- "Justificatory and context-setting recitals"
- "Commencement and temporal provisions"
- "Procedural safeguards and consultation requirements"
- "Scope-defining and definitional provisions"

BAD name examples (too thematic, too long):
- "Foundational purposes and cooperative architecture of EU FDI screening"
- "Confidentiality rules in foreign subsidy investigations"

Representative segments:
{desc_list}

Reply ONLY in this JSON format (no backticks):
{{"nome": "Layer Name", "descrizione": "Description in 2-3 sentences."}}"""


def build_prompt_layer_ranking(layer_records):
    layers_text = '\n'.join([
        f"  cluster_{r['cluster_id']}: {r['layer_name']} — {r['layer_description'][:120]}"
        for r in layer_records
    ])
    return f"""You are an expert in European law.

Below are functional layers found in a corpus of EU legal acts.
Rank them from most foundational (1 = constitutional basis, enabling norms, 
definitions) to most technical and operational (n = filing rules, timing, 
cross-references).

Layers:
{layers_text}

Reply ONLY with a JSON array of cluster_ids in order from most foundational 
to most technical (no other text, no backticks):
[cluster_id_1, cluster_id_2, ..., cluster_id_n]"""


# ── Fase B.2 — Naming ─────────────────────────────────────────────────────────

if os.path.exists(LAYER_MAPPING_FILE):
    layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)
    layer_records = layer_mapping_df.to_dict('records')
    print(f"Checkpoint trovato: {len(layer_records)} layer già nominati e ordinati.")
else:

    valid_cluster_ids = sorted([cid for cid in set(cluster_labels) if cid != -1])
    layer_records = []

    for cluster_id in valid_cluster_ids:
        repr_descs = get_representative_descriptions(segs_valid, cluster_id)
        prompt = build_prompt_layer_naming(repr_descs)
        response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_B2)

        nome        = f'Layer_{cluster_id}'
        descrizione = ''
        if status == 'ok':
            try:
                clean   = response_text.replace('```json', '').replace('```', '').strip()
                parsed  = json.loads(clean)
                nome    = parsed.get('nome', nome)
                descrizione = parsed.get('descrizione', '')
            except json.JSONDecodeError:
                nome = response_text[:80].strip()
                print(f"  cluster_{cluster_id}: JSON non valido, uso raw text come nome")

        layer_records.append({
            'cluster_id':        cluster_id,
            'layer_rank':        None,  # assegnato dopo
            'layer_name':        nome,
            'layer_description': descrizione,
            'n_segments':        int((segs_valid['cluster_id'] == cluster_id).sum()),
            'repr_descriptions': json.dumps(repr_descs, ensure_ascii=False),
            'llm_status':        status,
        })
        print(f"  cluster_{cluster_id:>3} → '{nome}'")
        time.sleep(LLM_DELAY_SECONDS)


    # ── Fase B.3 — Ranking LLM ────────────────────────────────────────────────────

    print("\nOrdinamento gerarchico via LLM...")
    prompt_rank = build_prompt_layer_ranking(layer_records)
    response_rank, status_rank = call_llm(client, prompt_rank, 200)

    if status_rank == 'ok':
        try:
            clean        = response_rank.replace('```json', '').replace('```', '').strip()
            ordered_ids  = json.loads(clean)
            ordered_ids = [int(str(x).replace('cluster_', '')) for x in ordered_ids]
            llm_rank     = {cid: rank + 1 for rank, cid in enumerate(ordered_ids)}
            # Fallback per cluster_id non restituiti dall'LLM
            missing = [r['cluster_id'] for r in layer_records if r['cluster_id'] not in llm_rank]
            for i, cid in enumerate(missing):
                llm_rank[cid] = len(ordered_ids) + i + 1
                print(f"  Warning: cluster_{cid} non nel ranking LLM, appeso in fondo")
        except (json.JSONDecodeError, TypeError) as e:
            print(f"  Ranking LLM non valido ({e}) — fallback su dimensione cluster")
            sorted_by_size = sorted(layer_records, key=lambda r: r['n_segments'], reverse=True)
            llm_rank = {r['cluster_id']: rank + 1 for rank, r in enumerate(sorted_by_size)}
    else:
        print("  Ranking LLM fallito — fallback su dimensione cluster")
        sorted_by_size = sorted(layer_records, key=lambda r: r['n_segments'], reverse=True)
        llm_rank = {r['cluster_id']: rank + 1 for rank, r in enumerate(sorted_by_size)}

    for r in layer_records:
        r['layer_rank'] = llm_rank[r['cluster_id']]


# ── Salvataggio e stampa ──────────────────────────────────────────────────────

layer_mapping_df = pd.DataFrame(layer_records).sort_values('layer_rank')
layer_mapping_df.to_csv(LAYER_MAPPING_FILE, index=False)

print()
print("=" * 60)
print("LAYER TROVATI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    print(f"  [{row['layer_rank']}] {row['layer_name']}")
    print(f"      {str(row['layer_description'])}")
    print(f"      N segmenti: {row['n_segments']:,}")
    print()

Checkpoint trovato: 5 layer già nominati e ordinati.

LAYER TROVATI
  [1] Scope-setting and framework clauses
      These provisions sit between the basic enabling framework and the detailed operational rules. They delimit when the regime applies, allocate institutional roles, and channel later implementation through references to follow-on acts, consultation steps, reporting duties, or internal procedures.
      N segmenti: 137

  [2] Procedural implementing provisions
      These provisions prescribe concrete administrative or evidentiary steps within an already established legal procedure. They do not create the underlying power, scope, or principle, but operationalize it through specific actions such as recording, transmitting, acknowledging, disclosing, or meeting deadlines.
      N segmenti: 282

  [3] Technical implementing provisions
      These segments are narrowly operative rules that translate a broader framework into concrete, case-specific steps, deadlines, evidentiary re

## 7. Fase C — Distribuzione Percentuale per Articolo (LLM 3)

Con i layer noti, per ogni **articolo** di ogni atto l'LLM produce una distribuzione
percentuale del contenuto tra i layer emersi.

Il risultato per ogni atto è la matrice **articoli × layer** (valori = %) che alimenta
la heatmap nell'applicazione.

> **Perché solo gli articoli?** I considerando hanno funzione giustificativa e retorica:
> spesso coprono più livelli intenzionalmente per costruire l'argomentazione legale.
> La varianza dei considerando riflette struttura retorica, non patologia.
> Gli articoli hanno funzione prescrittiva — la loro ibridità è il segnale diagnostico.

In [17]:
# Ricarica layer mapping (può essere eseguita anche senza rieseguire B)
layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)
layer_list = [
    {
        'rank':        int(row['layer_rank']),
        'name':        row['layer_name'],
        'description': row['layer_description'],
    }
    for _, row in layer_mapping_df.sort_values('layer_rank').iterrows()
]
layer_names = [l['name'] for l in layer_list]

# Colonne CSV safe (senza spazi/slash)
def to_col(name):
    return 'pct__' + name.replace(' ', '_').replace('/', '_')[:50]

pct_cols     = [to_col(n) for n in layer_names]
col_to_layer = {to_col(n): n for n in layer_names}

# Solo gli articoli
articles_df = segments_df[segments_df['tipo'] == 'articolo'].copy()

print(f"Layer trovati: {len(layer_list)}")
for l in layer_list:
    print(f"  [{l['rank']}] {l['name']}")
print()
print(f"Articoli da classificare: {len(articles_df):,}")
print(f"Atti coinvolti:           {articles_df['celex'].nunique():,}")

Layer trovati: 5
  [1] Scope-setting and framework clauses
  [2] Procedural implementing provisions
  [3] Technical implementing provisions
  [4] Procedural derogation clauses
  [5] Operational procedural micro-rules

Articoli da classificare: 848
Atti coinvolti:           19


In [18]:
def build_prompt_percentage_distribution(testo, identificatore, layer_list, tema):
    layers_desc = '\n'.join([
        f"  {l['rank']}. {l['name']}: {l['description']}"
        for l in layer_list
    ])
    layer_keys = ', '.join([f'"{l["name"]}"' for l in layer_list])

    return f"""You are an expert in European law.

Read the article below and distribute its content as a percentage across 
the following hierarchical normative layers. Each layer represents a distinct 
functional role in the regulatory hierarchy — assign percentages based on 
what normative function each part of the article performs, not on its topic.

Layers:
{layers_desc}

Rules:
- Percentages must sum to exactly 100.
- Assign 0 to layers not present in the article.
- If the article mixes layers, reflect the actual proportions.

Article {identificatore}:
{testo}

Reply ONLY with valid JSON (no other text, no backticks):
{{{layer_keys}}}"""


def parse_percentage_response(response_text, layer_names):
    """
    Parsa la risposta JSON e normalizza a somma 100.
    Restituisce None se il parsing fallisce.
    """
    try:
        clean  = response_text.replace('```json', '').replace('```', '').strip()
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        return None

    values = {name: float(parsed.get(name, 0)) for name in layer_names}
    total  = sum(values.values())
    if total <= 0:
        return None
    if abs(total - 100) > 5:   # normalizza se la somma si discosta
        values = {k: v / total * 100 for k, v in values.items()}
    return values


print("Funzioni Fase C definite.")

Funzioni Fase C definite.


In [19]:
%%time
# ── Gestione checkpoint ────────────────────────────────────────────────────────
if os.path.exists(HEATMAP_CKPT_FILE):
    heatmap_done = pd.read_csv(HEATMAP_CKPT_FILE)
    done_seg_ids = set(heatmap_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_seg_ids):,} articoli già classificati.")
else:
    heatmap_done = pd.DataFrame()
    done_seg_ids = set()
    print("Nessun checkpoint heatmap — si parte da zero.")

articles_todo = articles_df[~articles_df['segment_id'].isin(done_seg_ids)].copy()
print(f"Articoli da classificare: {len(articles_todo):,}")
print()

new_rows = []
n_ok_c = n_error_c = 0
total_c = len(articles_todo)

for i, (_, art) in enumerate(articles_todo.iterrows()):

    prompt = build_prompt_percentage_distribution(
        testo          = art['testo'],
        identificatore = art['identificatore'],
        layer_list     = layer_list,
        tema           = TEMA_DESCRIZIONE,
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_C)

    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, layer_names)
        if distribution is None:
            status = 'parse_error'

    row = {
        'segment_id':  art['segment_id'],
        'celex':       art['celex'],
        'node_id':     art['node_id'],
        'articolo_id': art['identificatore'],
        'llm_status':  status,
    }
    for col, name in zip(pct_cols, layer_names):
        row[col] = round(distribution[name], 2) if distribution else 0.0

    if distribution:
        n_ok_c += 1
    else:
        n_error_c += 1

    new_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total_c:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([heatmap_done, batch], ignore_index=True) if not heatmap_done.empty else batch
        combined.to_csv(HEATMAP_CKPT_FILE, index=False)
        print(f"  [{i+1:>5}/{total_c}]  {(i+1)/total_c*100:5.1f}%   ok: {n_ok_c}   errori: {n_error_c}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE C — ok: {n_ok_c:,}   errori: {n_error_c:,}")
print("=" * 50)

Checkpoint trovato: 848 articoli già classificati.
Articoli da classificare: 0


FASE C — ok: 0   errori: 0
CPU times: total: 15.6 ms
Wall time: 12.2 ms


In [20]:
# Salva heatmap finale
heatmap_final = pd.read_csv(HEATMAP_CKPT_FILE)
heatmap_final.to_csv(NODES_HEATMAP_FILE, index=False)

print(f"Salvato: {NODES_HEATMAP_FILE}")
print(f"Righe: {len(heatmap_final):,}  |  Colonne pct: {pct_cols}")
print()

ok_mask = heatmap_final['llm_status'] == 'ok'
print("Distribuzione media % per layer (articoli ok):")
for col in pct_cols:
    mean_pct = heatmap_final.loc[ok_mask, col].mean()
    bar = '█' * int(mean_pct / 2)
    print(f"  {col_to_layer[col][:45]:.<46} {mean_pct:5.1f}%  {bar}")

Salvato: ..\data\output\fdi_screening\nodes_heatmap.csv
Righe: 848  |  Colonne pct: ['pct__Scope-setting_and_framework_clauses', 'pct__Procedural_implementing_provisions', 'pct__Technical_implementing_provisions', 'pct__Procedural_derogation_clauses', 'pct__Operational_procedural_micro-rules']

Distribuzione media % per layer (articoli ok):
  Scope-setting and framework clauses...........  27.3%  █████████████
  Procedural implementing provisions............  16.7%  ████████
  Technical implementing provisions.............  18.3%  █████████
  Procedural derogation clauses.................  14.9%  ███████
  Operational procedural micro-rules............  22.9%  ███████████


## 7b. Fase C2 — Entropia Lamfalussy per Articolo

Calcola lo **score di ibridità Lamfalussy** partendo dall'output di Fase A2.
Nessuna chiamata LLM — è una trasformazione puramente computazionale.

Per ogni atto produce:
- `hybridity_lamf_score` — entropia media degli articoli su L1–L4 (metrica principale)
- `dominant_lamf` — livello Lamfalussy dominante

**Input**: `segments_lamfalussy.csv` (tutti i segmenti, filtrati agli articoli)

**Output**: `nodes_lamfalussy.csv` (articoli con distribuzione L1–L4, per le visualizzazioni)

In [21]:
# ── Fase C2: entropia Lamfalussy per articolo (nessun LLM) ────────────────────
if not os.path.exists(SEGMENTS_LAMF_FILE):
    print(f"  {SEGMENTS_LAMF_FILE} non trovato — esegui prima Fase A2 (sezione 4b).")
else:
    segments_lamf = pd.read_csv(SEGMENTS_LAMF_FILE)

    # Filtra ai soli articoli (come Fase C sui layer emersi)
    articles_lamf = segments_lamf[
        (segments_lamf['llm_status'] == 'ok') &
        (segments_lamf['tipo'] == 'articolo')
    ].copy()

    # Assicura che le colonne LAMF_COLS esistano
    for col in LAMF_COLS:
        if col not in articles_lamf.columns:
            articles_lamf[col] = 0.0

    # Rinomina come nodes_lamfalussy.csv (compatibilità con le celle di visualizzazione)
    # aggiunge articolo_id per allineamento con heatmap_ok
    articles_lamf['articolo_id'] = articles_lamf['identificatore']
    articles_lamf.to_csv(NODES_LAMFALUSSY_FILE, index=False)

    print(f"Salvato: {NODES_LAMFALUSSY_FILE}")
    print(f"Articoli: {len(articles_lamf):,}  |  Atti: {articles_lamf['celex'].nunique():,}")
    print()
    print("Distribuzione media % per livello Lamfalussy (articoli):")
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
        mean_pct = articles_lamf[col].mean()
        bar      = '█' * int(mean_pct / 2)
        print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")


Salvato: ..\data\output\fdi_screening\nodes_lamfalussy.csv
Articoli: 848  |  Atti: 19

Distribuzione media % per livello Lamfalussy (articoli):
  L1 — Framework principles.....................   2.9%  █
  L2 — Implementing measures....................  70.7%  ███████████████████████████████████
  L3 — Coordination and convergence.............   4.8%  ██
  L4 — Enforcement and monitoring...............  21.6%  ██████████


## 8. Fase D — Score di Ibridità per Atto

Lo score di ibridità misura quanto gli articoli di un atto variano nel loro livello
gerarchico. Un atto **puro** ha tutti gli articoli concentrati sullo stesso layer.
Un atto **ibrido** ha articoli che spaziano su layer molto diversi.

**Metrica: entropia di Shannon normalizzata per articolo**

$$H(a) = -\sum_{l} p_{al} \log_2(p_{al} + \varepsilon)$$

Lo score dell'atto è la **media delle entropie dei propri articoli**.
Zero = tutti gli articoli sono monofunzionali. Uno = distribuzione uniforme su tutti i layer.

In [22]:
def entropy_norm(row, pct_cols):
    """Entropia di Shannon normalizzata (0=puro, 1=uniforme)."""
    eps   = 1e-9
    probs = np.array([float(row.get(c, 0)) for c in pct_cols]) / 100.0
    probs = np.clip(probs, 0, 1)
    s     = probs.sum()
    if s < eps:
        return 0.0
    probs = probs / s
    raw   = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(pct_cols)) if len(pct_cols) > 1 else 1.0
    return float(raw / maxH)


def dominant_layer(row, pct_cols, col_to_layer):
    best = max(pct_cols, key=lambda c: float(row.get(c, 0)))
    return col_to_layer.get(best, best)


# ── Articoli classificati correttamente (layer emersi) ────────────────────────
heatmap_ok = heatmap_final[heatmap_final['llm_status'] == 'ok'].copy()

# Entropia su layer emersi — metrica secondaria
heatmap_ok['entropy_layers'] = heatmap_ok.apply(
    lambda r: entropy_norm(r, pct_cols), axis=1
)
heatmap_ok['dominant_layer'] = heatmap_ok.apply(
    lambda r: dominant_layer(r, pct_cols, col_to_layer), axis=1
)

# ── Lamfalussy: definizione colonne (self-contained) ──────────────────────────
# Definite qui per garantire disponibilità anche se la cella 7b non è stata eseguita
_LAMF_KEYS_D = ['L1', 'L2', 'L3', 'L4']
_LAMF_COLS_D = [f'lamf_{k}' for k in _LAMF_KEYS_D]

# Merge entropia Lamfalussy in heatmap_ok ────────────────────────────────────
if os.path.exists(NODES_LAMFALUSSY_FILE):
    lamfalussy_final = pd.read_csv(NODES_LAMFALUSSY_FILE)
    lamf_ok = lamfalussy_final[lamfalussy_final['llm_status'] == 'ok'].copy()

    # Assicura che le colonne Lamfalussy esistano (gestisce run parziali)
    for col in _LAMF_COLS_D:
        if col not in lamf_ok.columns:
            lamf_ok[col] = 0.0

    lamf_ok['lamf_entropy'] = lamf_ok.apply(
        lambda r: entropy_norm(r, _LAMF_COLS_D), axis=1
    )
    lamf_ok['dominant_lamf'] = lamf_ok.apply(
        lambda r: max(_LAMF_COLS_D, key=lambda c: float(r.get(c, 0))).replace('lamf_', ''),
        axis=1
    )

    # Merge per-articolo in heatmap_ok: disponibile per le celle di visualizzazione
    heatmap_ok = heatmap_ok.merge(
        lamf_ok[['segment_id', 'lamf_entropy', 'dominant_lamf'] + _LAMF_COLS_D],
        on='segment_id', how='left'
    )
    heatmap_ok['lamf_entropy'] = heatmap_ok['lamf_entropy'].fillna(0.0)
    lamf_available = True
    print(f"Lamfalussy mergato in heatmap_ok: {lamf_ok['lamf_entropy'].notna().sum():,} articoli")
else:
    heatmap_ok['lamf_entropy']  = heatmap_ok['entropy_layers']   # fallback
    heatmap_ok['dominant_lamf'] = heatmap_ok['dominant_layer']
    lamf_available = False
    print(f"  ⚠ {NODES_LAMFALUSSY_FILE} non trovato — uso layer emersi come fallback per lamf_entropy")

# ── Aggregazione per atto ──────────────────────────────────────────────────────
def agg_atto(group):
    return pd.Series({
        # ── primario: Lamfalussy (o fallback su layer emersi) ─────────────────
        'hybridity_score':           group['lamf_entropy'].mean(),
        'hybridity_std':             group['lamf_entropy'].std(),
        'hybridity_max':             group['lamf_entropy'].max(),
        'most_hybrid_article':       (group.nlargest(1, 'lamf_entropy')['articolo_id'].iloc[0]
                                       if len(group) > 0 else ''),
        'dominant_lamf':             (group['dominant_lamf'].mode().iloc[0]
                                       if len(group) > 0 else ''),
        # ── secondario: layer emersi ──────────────────────────────────────────
        'hybridity_layers_score':    group['entropy_layers'].mean(),
        'hybridity_layers_std':      group['entropy_layers'].std(),
        'hybridity_layers_max':      group['entropy_layers'].max(),
        'dominant_layer':            (group['dominant_layer'].mode().iloc[0]
                                       if len(group) > 0 else ''),
        'dominant_layer_pct':        (group['dominant_layer'].value_counts().iloc[0] / len(group) * 100
                                       if len(group) > 0 else 0.0),
        'n_articles':                len(group),
    })

hybridity_df = heatmap_ok.groupby('celex').apply(agg_atto).reset_index()

# Aggiunge metadati dal nodo originale
meta_cols  = [c for c in ['Id', 'Label', 'title', 'LegalType', 'Year', 'PipelineLevel']
              if c in nodes.columns]
nodes_meta = nodes[meta_cols].copy()
nodes_meta = nodes_meta.rename(columns={'Label': 'celex'}) if 'Label' in nodes_meta.columns else nodes_meta

hybridity_df = hybridity_df.merge(nodes_meta, on='celex', how='left')
hybridity_df = hybridity_df.drop_duplicates(subset=['celex'], keep='first')
hybridity_df = hybridity_df.sort_values('hybridity_score', ascending=False)
hybridity_df.to_csv(NODES_HYBRIDITY_FILE, index=False)

print(f"Salvato: {NODES_HYBRIDITY_FILE}")
print(f"Atti analizzati: {len(hybridity_df):,}")
print()
score_label = "Lamfalussy" if lamf_available else "Layer emersi (fallback)"
desc = hybridity_df['hybridity_score'].describe()
print(f"Statistiche hybridity_score ({score_label}):")
print(f"  Media:   {desc['mean']:.4f}")
print(f"  Mediana: {desc['50%']:.4f}")
print(f"  Max:     {desc['max']:.4f}")
print(f"  Std:     {desc['std']:.4f}")
print()
print("Top 5 atti più ibridi:")
for _, row in hybridity_df.head(5).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {celex_label:<20}  score={row['hybridity_score']:.3f}  "
          f"dominant={row['dominant_lamf']}  "
          f"(layers={row['hybridity_layers_score']:.3f})")


Lamfalussy mergato in heatmap_ok: 848 articoli
Salvato: ..\data\output\fdi_screening\nodes_hybridity.csv
Atti analizzati: 19

Statistiche hybridity_score (Lamfalussy):
  Media:   0.1143
  Mediana: 0.1538
  Max:     0.3110
  Std:     0.1025

Top 5 atti più ibridi:
  32005R0184            score=0.311  dominant=L4  (layers=0.424)
  32020D1502            score=0.215  dominant=L2  (layers=0.406)
  32019R0452            score=0.215  dominant=L2  (layers=0.389)
  32016R1013            score=0.210  dominant=L2  (layers=0.319)
  32012R1219            score=0.204  dominant=L2  (layers=0.434)


## 9. Diagnostica e Visualizzazioni

In [23]:
print("=" * 60)
print("LAYER EMERSI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    n    = row['n_segments']
    pct  = n / len(segs_valid) * 100
    bar  = '█' * int(pct)
    print(f"[{row['layer_rank']:>2}] {row['layer_name']}")
    print(f"     Segmenti: {n:,}  ({pct:.1f}%)  {bar}")
    print(f"     {str(row['layer_description'])[:110]}")
    print()

LAYER EMERSI
[ 1] Scope-setting and framework clauses
     Segmenti: 137  (16.2%)  ████████████████
     These provisions sit between the basic enabling framework and the detailed operational rules. They delimit whe

[ 2] Procedural implementing provisions
     Segmenti: 282  (33.3%)  █████████████████████████████████
     These provisions prescribe concrete administrative or evidentiary steps within an already established legal pr

[ 3] Technical implementing provisions
     Segmenti: 100  (11.8%)  ███████████
     These segments are narrowly operative rules that translate a broader framework into concrete, case-specific st

[ 4] Procedural derogation clauses
     Segmenti: 95  (11.2%)  ███████████
     These provisions sit at the technical, operational end of the hierarchy and do not establish broad principles 

[ 5] Operational procedural micro-rules
     Segmenti: 108  (12.7%)  ████████████
     These provisions sit at the lowest, most concrete level of the hierarchy: they specify 

In [24]:
print("=" * 60)
print("TOP 10 ATTI PIÙ IBRIDI")
print("=" * 60)
print()
for _, row in hybridity_df.head(10).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {row['hybridity_score']:.4f}  {celex_label}")
    print(f"           Dominant Lamfalussy: {row.get('dominant_lamf','-')}  |  Layer emerso: {row.get('dominant_layer','-')} ({row.get('dominant_layer_pct',0):.0f}%)")
    print(f"           Art: {row['n_articles']}  |  Più ibrido: {row['most_hybrid_article']}")
    print(f"           {str(row.get('title',''))[:70]}")
    print()

TOP 10 ATTI PIÙ IBRIDI

  0.3110  32005R0184
           Dominant Lamfalussy: L4  |  Layer emerso: Procedural implementing provisions (38%)
           Art: 21  |  Più ibrido: 4_p4
           REGULATION (EC) No 184/2005 OF THE EUROPEAN PARLIAMENT AND OF THE COUN

  0.2151  32020D1502
           Dominant Lamfalussy: L2  |  Layer emerso: Scope-setting and framework clauses (35%)
           Art: 40  |  Più ibrido: 3_p2
           COMMISSION DECISION (EU) 2020/1502 of 15 October 2020 laying down inte

  0.2151  32019R0452
           Dominant Lamfalussy: L2  |  Layer emerso: Scope-setting and framework clauses (41%)
           Art: 94  |  Più ibrido: 6_p2
           REGULATION (EU) 2019/452 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL

  0.2105  32016R1013
           Dominant Lamfalussy: L2  |  Layer emerso: Scope-setting and framework clauses (56%)
           Art: 32  |  Più ibrido: 1_p31
           REGULATION (EU) 2016/1013 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCI

  0.2040  32012R1219

In [25]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Istogramma ibridità
ax1 = axes[0]
ax1.hist(hybridity_df['hybridity_score'], bins=20, edgecolor='black', color='steelblue')
ax1.axvline(hybridity_df['hybridity_score'].median(), color='red', linestyle='--',
            label=f"Mediana = {hybridity_df['hybridity_score'].median():.3f}")
ax1.set_xlabel('Hybridity Score')
ax1.set_ylabel('N atti')
ax1.set_title('Distribuzione Score di Ibridità')
ax1.legend()

# Top 15 atti
ax2 = axes[1]
top15  = hybridity_df.head(15)
labels = top15['celex'].apply(lambda x: str(x)[:14]).tolist()
ax2.barh(range(len(top15)), top15['hybridity_score'], color='tomato')
ax2.set_yticks(range(len(top15)))
ax2.set_yticklabels(labels, fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel('Hybridity Score')
ax2.set_title('Top 15 Atti più Ibridi')

plt.tight_layout()

fig_dir  = os.path.join(output_path, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, 'hybridity_distribution.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figura salvata: {fig_path}")

Figura salvata: ..\data\output\fdi_screening\figures\hybridity_distribution.png


C:\Users\claud\AppData\Local\Temp\ipykernel_19672\3557925531.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import numpy as np

CELEX_TARGET = '32019R0452'

# ── Heatmap: layer emersi (colonne invariate) ─────────────────────────────────
atto = heatmap_final[
    (heatmap_final['celex'] == CELEX_TARGET) &
    (heatmap_final['llm_status'] == 'ok')
].copy()

atto['articolo_base'] = atto['articolo_id'].str.extract(r'^(\d+)')[0]
atto_agg = atto.groupby('articolo_base')[pct_cols].mean()
atto_agg.index = atto_agg.index.astype(int)
atto_agg = atto_agg.sort_index()
atto_agg.index = atto_agg.index.astype(str)

article_ids = atto_agg.index.tolist()
matrix      = atto_agg.values / 100.0

# ── Entropia sidebar: Lamfalussy (da heatmap_ok già mergato in Fase D) ─────────
# heatmap_ok contiene 'lamf_entropy' per ogni articolo (o fallback su layer emersi)
_lamf_cols_viz = [c for c in heatmap_ok.columns if c.startswith('lamf_L')]
if _lamf_cols_viz:
    # Aggregazione per articolo_base sulle colonne L1-L4
    lamf_atto_ok = heatmap_ok[heatmap_ok['celex'] == CELEX_TARGET].copy()
    lamf_atto_ok['articolo_base'] = lamf_atto_ok['articolo_id'].str.extract(r'^(\d+)')[0]
    lamf_agg = lamf_atto_ok.groupby('articolo_base')[_lamf_cols_viz].mean()
    lamf_agg.index = lamf_agg.index.astype(int)
    lamf_agg = lamf_agg.sort_index()
    lamf_agg.index = lamf_agg.index.astype(str)
    lamf_agg = lamf_agg.reindex(article_ids).fillna(0)
    entropy_col = np.array([
        entropy_norm(lamf_agg.iloc[i], _lamf_cols_viz)
        for i in range(len(lamf_agg))
    ])
    entropy_label = 'H (Lamfalussy)'
    entropy_cb_label = 'H  (0=puro, 1=ibrido)\nL1–L4 Lamfalussy'
else:
    # Fallback: layer emersi (se Fase C-Lamfalussy non è stata eseguita)
    entropy_col = np.array([
        entropy_norm(atto_agg.iloc[i], pct_cols)
        for i in range(len(atto_agg))
    ])
    entropy_label = 'H (layers)'
    entropy_cb_label = 'H  (0=puro, 1=ibrido)\nLayer emersi (fallback)'
    print("  ⚠ Lamfalussy non disponibile — uso layer emersi come fallback")

entropy_col = np.clip(entropy_col, 0, 1)

n_articles = len(article_ids)
n_layers   = len(pct_cols)

layer_names_full = [col_to_layer.get(c, c) for c in pct_cols]
short_names = [
    n.replace(' and ', '\n& ')
     .replace(' clauses', '')
     .replace(' provisions', '')
     .replace(' rules', '')
     .replace(' conditions', '')
    for n in layer_names_full
]

# ── Layout ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(max(20, n_layers * 1.6), max(10, n_articles * 0.5)))
gs  = gridspec.GridSpec(
    1, 2,
    width_ratios=[n_layers, 1],
    wspace=0.01,
    left=0.10, right=0.90, top=0.92, bottom=0.22
)
ax_main = fig.add_subplot(gs[0])
ax_ent  = fig.add_subplot(gs[1])

# ── Heatmap layer emersi ───────────────────────────────────────────────────────
im_layers = ax_main.imshow(
    matrix, aspect='auto',
    cmap=plt.cm.Blues, vmin=0, vmax=1,
    interpolation='nearest'
)

for i in range(n_articles):
    for j in range(n_layers):
        val   = matrix[i, j]
        color = 'white' if val > 0.50 else ('lightgray' if val < 0.05 else 'black')
        ax_main.text(j, i, f'{val*100:.0f}%', ha='center', va='center',
                     fontsize=8.5, color=color,
                     fontweight='bold' if val > 0.35 else 'normal')

for y in range(1, n_articles):
    ax_main.axhline(y - 0.5, color='white', linewidth=0.3)
for x in range(1, n_layers):
    ax_main.axvline(x - 0.5, color='white', linewidth=0.3)

ax_main.set_xticks(range(n_layers))
ax_main.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax_main.set_yticks(range(n_articles))
ax_main.set_yticklabels(article_ids, fontsize=8)
ax_main.set_xlabel('Normative Layer', fontsize=10, labelpad=10)
ax_main.set_ylabel('Article', fontsize=10)
ax_main.set_title(
    f'Heatmap — {CELEX_TARGET}\n'
    f'Per-article normative layer distribution + Shannon entropy (H)',
    fontsize=11, pad=12
)

# ── Barra entropia Lamfalussy ─────────────────────────────────────────────────
im_ent = ax_ent.imshow(
    entropy_col.reshape(-1, 1), aspect='auto',
    cmap=plt.cm.RdYlGn_r, vmin=0, vmax=1,
    interpolation='nearest'
)

for i in range(n_articles):
    h_val = entropy_col[i]
    color = 'white' if h_val > 0.55 else 'black'
    ax_ent.text(0, i, f'{h_val:.2f}', ha='center', va='center',
                fontsize=8, fontweight='bold', color=color)

for y in range(1, n_articles):
    ax_ent.axhline(y - 0.5, color='white', linewidth=0.3)

ax_ent.set_xticks([0])
ax_ent.set_xticklabels([entropy_label], fontsize=7)   # fontsize come kwarg, non in lista
ax_ent.set_yticks(range(n_articles))
ax_ent.set_yticklabels([])
ax_ent.tick_params(left=False)

# ── Colorbars ─────────────────────────────────────────────────────────────────
cb1 = fig.colorbar(im_layers, ax=ax_main, shrink=0.5, pad=0.01)
cb1.set_label('% content in layer', fontsize=8)
cb1.ax.tick_params(labelsize=7)

cb2 = fig.colorbar(im_ent, ax=ax_ent, shrink=0.5, pad=0.12)
cb2.set_label(entropy_cb_label, fontsize=8)
cb2.ax.tick_params(labelsize=7)

fig_path = os.path.join(output_path, 'figures', f'heatmap_entropy_{CELEX_TARGET}.png')
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvato: {fig_path}")


Salvato: ..\data\output\fdi_screening\figures\heatmap_entropy_32019R0452.png


C:\Users\claud\AppData\Local\Temp\ipykernel_19672\2962906498.py:143: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9b. Heatmap Aggregata — Tutti gli Atti

Una singola heatmap con **tutti gli atti** sull'asse Y e i **layer normativi** sull'asse X.

Ogni cella mostra la **percentuale media** del contenuto dell'atto in quel layer,
calcolata come media delle distribuzioni di tutti gli articoli dell'atto.

Gli atti sono ordinati per **score di ibridità decrescente** (l'atto più ibrido in cima).

In [27]:
import matplotlib.pyplot as plt
import numpy as np

# ── 1. Aggregazione: media pct per atto ───────────────────────────────────────
act_means = (
    heatmap_ok
    .groupby('celex')[pct_cols]
    .mean()
    .reset_index()
)

act_means = act_means.merge(
    hybridity_df[['celex', 'hybridity_score', 'hybridity_layers_score',
                  'dominant_lamf', 'n_articles']],
    on='celex', how='left'
)

# Ordina per hybridity_score (= Lamfalussy se disponibile, layer emersi altrimenti)
act_means = act_means.sort_values('hybridity_score', ascending=False).reset_index(drop=True)

# ── 2. Matrice e label ────────────────────────────────────────────────────────
matrix     = act_means[pct_cols].values / 100.0
act_ids    = act_means['celex'].tolist()
hyb_scores = act_means['hybridity_score'].tolist()
lay_scores = act_means['hybridity_layers_score'].tolist()
n_articles = act_means['n_articles'].tolist()

layer_names = [col_to_layer.get(c, c) for c in pct_cols]
short_names = [
    n.replace(' and ', '\n& ')
     .replace(' clauses', '')
     .replace(' provisions', '')
     .replace(' rules', '')
     .replace(' conditions', '')
    for n in layer_names
]

y_labels = [
    f"{celex}  (H={h:.3f}, H_layers={hl:.3f}, n={int(n)})"
    for celex, h, hl, n in zip(act_ids, hyb_scores, lay_scores, n_articles)
]

# ── 3. Plot ───────────────────────────────────────────────────────────────────
n_acts   = len(act_ids)
n_layers = len(pct_cols)

fig, ax = plt.subplots(figsize=(max(18, n_layers * 1.5), max(8, n_acts * 0.55)))

im = ax.imshow(matrix, aspect='auto', cmap=plt.cm.Blues, vmin=0, vmax=1,
               interpolation='nearest')

for i in range(n_acts):
    for j in range(n_layers):
        val   = matrix[i, j]
        color = 'white' if val > 0.50 else ('lightgray' if val < 0.05 else 'black')
        ax.text(j, i, f'{val*100:.0f}%', ha='center', va='center',
                fontsize=8.5, color=color,
                fontweight='bold' if val > 0.35 else 'normal')

for y in range(1, n_acts):
    ax.axhline(y - 0.5, color='white', linewidth=1.0 if y % 5 == 0 else 0.3)
for x in range(1, n_layers):
    ax.axvline(x - 0.5, color='white', linewidth=0.3)

ax.set_xticks(range(n_layers))
ax.set_xticklabels(short_names, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(n_acts))
ax.set_yticklabels(y_labels, fontsize=8, family='monospace')
ax.set_xlabel('Normative Layer  (emerged from data)', fontsize=10, labelpad=12)
ax.set_ylabel('Legal Act  (sorted by H Lamfalussy ↓)', fontsize=10, labelpad=10)
ax.set_title(
    f'Aggregate Heatmap — All {n_acts} Acts × {n_layers} Normative Layers\n'
    'Mean % of articles per emerged layer  |  sorted by Shannon entropy H (Lamfalussy L1–L4)',
    fontsize=12, pad=14
)

cb = plt.colorbar(im, ax=ax, label='Mean % content in layer', shrink=0.55, pad=0.02)
cb.ax.tick_params(labelsize=8)

plt.tight_layout()

fig_dir  = os.path.join(output_path, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, 'heatmap_all_acts.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvato: {fig_path}")
print(f"Atti: {n_acts}  |  Layer: {n_layers}  |  Matrice: {matrix.shape}")


Salvato: ..\data\output\fdi_screening\figures\heatmap_all_acts.png
Atti: 19  |  Layer: 5  |  Matrice: (19, 5)


C:\Users\claud\AppData\Local\Temp\ipykernel_19672\1348083093.py:86: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9c. Grafo 

In [28]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
import numpy as np

# ── Carica dati ───────────────────────────────────────────────────────────────
edges_df    = pd.read_csv(EDGES_FILE)
hyb         = hybridity_df[['celex', 'hybridity_score', 'n_articles', 'title']].copy()

# ── Costruisci grafo ──────────────────────────────────────────────────────────
G = nx.DiGraph()
for celex in hyb['celex']:
    G.add_node(celex)

for _, e in edges_df.iterrows():
    src = str(e.get('Source', e.get(':START_ID', '')))
    tgt = str(e.get('Target', e.get(':END_ID', '')))
    if src in G and tgt in G:
        G.add_edge(src, tgt)

print(f"Nodi: {G.number_of_nodes()}  |  Archi: {G.number_of_edges()}")

# ── Layout ────────────────────────────────────────────────────────────────────
pos = nx.spring_layout(G, k=2.5, seed=42)

# ── Colori per ibridità (verde → rosso) ───────────────────────────────────────
scores   = hyb.set_index('celex')['hybridity_score'].to_dict()
n_arts   = hyb.set_index('celex')['n_articles'].to_dict()
titles   = hyb.set_index('celex')['title'].to_dict()

node_list   = list(G.nodes())
score_vals  = np.array([scores.get(n, 0) for n in node_list])
score_norm  = (score_vals - score_vals.min()) / (score_vals.max() - score_vals.min() + 1e-9)

cmap        = plt.cm.RdYlGn_r   # verde=puro, rosso=ibrido
node_colors = [cmap(s) for s in score_norm]
node_sizes  = [max(300, n_arts.get(n, 2) * 4) for n in node_list]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 12))
ax.set_facecolor('#f8f8f8')
fig.patch.set_facecolor('#f8f8f8')

# Archi
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color='#aaaaaa', alpha=0.4,
    arrows=True, arrowsize=10,
    width=0.8,
    connectionstyle='arc3,rad=0.08',
)

# Nodi
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    nodelist=node_list,
    node_color=node_colors,
    node_size=node_sizes,
    linewidths=1.2,
    edgecolors='white',
)

# Etichette CELEX
nx.draw_networkx_labels(
    G, pos, ax=ax,
    labels={n: n for n in node_list},
    font_size=6.5,
    font_color='#222222',
    font_weight='bold',
)

# ── Colorbar ──────────────────────────────────────────────────────────────────
sm = plt.cm.ScalarMappable(cmap=cmap,
     norm=mcolors.Normalize(vmin=score_vals.min(), vmax=score_vals.max()))
sm.set_array([])
cb = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
cb.set_label('Hybridity Score', fontsize=10)
cb.ax.tick_params(labelsize=8)

ax.set_title(
    'FDI Screening — Rete Normativa\n'
    'Colore: ibridità (verde=puro, rosso=ibrido) | Dimensione: numero articoli',
    fontsize=13, pad=14
)
ax.axis('off')
plt.tight_layout()

fig_path = os.path.join(output_path, 'figures', 'network_hybridity.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvato: {fig_path}")

Nodi: 19  |  Archi: 17
Salvato: ..\data\output\fdi_screening\figures\network_hybridity.png


C:\Users\claud\AppData\Local\Temp\ipykernel_19672\1556060275.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Riepilogo Output

Verifica che tutti i file di output siano stati prodotti correttamente.

In [29]:
output_files = {
    'segments_descriptions.csv': SEGMENTS_DESC_FILE,
    'layer_mapping.csv':         LAYER_MAPPING_FILE,
    'nodes_heatmap.csv':         NODES_HEATMAP_FILE,
    'nodes_hybridity.csv':       NODES_HYBRIDITY_FILE,
    'segments_lamfalussy.csv':    SEGMENTS_LAMF_FILE,
    'nodes_lamfalussy.csv':       NODES_LAMFALUSSY_FILE,
}

print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

all_ok = True
for name, path in output_files.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        df_tmp  = pd.read_csv(path)
        print(f"  ✓ {name}")
        print(f"    Righe: {len(df_tmp):,}  |  Dim: {size_kb:.1f} KB")
        print(f"    Colonne: {list(df_tmp.columns)[:6]}{'...' if len(df_tmp.columns) > 6 else ''}")
    else:
        print(f"  ✗ {name} — FILE MANCANTE")
        all_ok = False
    print()

if all_ok:
    print("✓ Pipeline 04 completata.")
else:
    print("  Alcuni file mancano — rieseguire le celle corrispondenti.")

OUTPUT FILES
  ✓ segments_descriptions.csv
    Righe: 848  |  Dim: 555.2 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'testo_originale']...

  ✓ layer_mapping.csv
    Righe: 5  |  Dim: 18.5 KB
    Colonne: ['cluster_id', 'layer_rank', 'layer_name', 'layer_description', 'n_segments', 'repr_descriptions']...

  ✓ nodes_heatmap.csv
    Righe: 848  |  Dim: 68.1 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'articolo_id', 'llm_status', 'pct__Scope-setting_and_framework_clauses']...

  ✓ nodes_hybridity.csv
    Righe: 19  |  Dim: 9.3 KB
    Colonne: ['celex', 'hybridity_score', 'hybridity_std', 'hybridity_max', 'most_hybrid_article', 'dominant_lamf']...

  ✓ segments_lamfalussy.csv
    Righe: 848  |  Dim: 71.5 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'llm_status']...

  ✓ nodes_lamfalussy.csv
    Righe: 848  |  Dim: 76.1 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'llm_status']...

✓ Pip

## Fase E — Export per il frontend HTML

Genera `HEATMAPS` (dizionario `celex → articoli`) e patcha l'HTML self-contained
con tutti i dati della pipeline: layer labels, distribuzioni percentuali, campo `heatmap` nei nodi.

**Cambia `HTML_FILE`** per puntare al file corretto prima di eseguire.

In [32]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE E — Export HEATMAPS + distribuzione Lamfalussy per il frontend HTML
# ══════════════════════════════════════════════════════════════════════════════

HTML_FILE = os.path.join('..', 'fdi_network.html')   

import re, math, json as _json

# ── 1. Layer labels da layer_mapping.csv ──────────────────────────────────────
layer_df = pd.read_csv(LAYER_MAPPING_FILE).sort_values('layer_rank')
layer_names_raw = layer_df['layer_name'].tolist()

def split_label(name):
    s = name.replace('_', ' ')
    words = s.split()
    if len(words) == 1:
        return s, ''
    half = len(s) // 2
    pos, best_split = 0, len(words) // 2
    for i, w in enumerate(words[:-1]):
        pos += len(w) + 1
        if pos >= half:
            best_split = i + 1
            break
    return ' '.join(words[:best_split]), ' '.join(words[best_split:])

l1_arr, l2_arr = [], []
for name in layer_names_raw:
    a, b = split_label(name)
    l1_arr.append(a)
    l2_arr.append(b)

print(f"Layer ({len(layer_names_raw)}):")
for i, name in enumerate(layer_names_raw):
    print(f"  {i:2d}  '{l1_arr[i]}' / '{l2_arr[i]}'")

# ── 2. HEATMAPS dict da nodes_heatmap.csv ────────────────────────────────────
hm_df = pd.read_csv(NODES_HEATMAP_FILE)
id_col = 'id' if 'id' in hm_df.columns else 'celex'
pct_cols_hm = [c for c in hm_df.columns if c.startswith('pct__')]
n_layers = len(pct_cols_hm)

HEATMAPS = {}
for act_id, grp in hm_df.groupby(id_col):
    ok_grp = grp[grp['llm_status'] == 'ok'].copy()
    if ok_grp.empty:
        continue
    articles = []
    for _, row in ok_grp.iterrows():
        vals = [float(row[c]) for c in pct_cols_hm]
        H = 0.0
        for v in vals:
            p = v / 100.0
            if p > 1e-9:
                H -= p * math.log2(p)
        H_norm = round(H / math.log2(n_layers), 3) if n_layers > 1 else 0.0
        articles.append({'id': str(row['articolo_id']), 'H': H_norm,
                         'vals': [round(v, 1) for v in vals]})
    if articles:
        HEATMAPS[str(act_id)] = articles

print(f"\nHEATMAPS: {len(HEATMAPS)} atti  |  "
      f"{sum(len(v) for v in HEATMAPS.values())} articoli totali")

# ── 3. Distribuzione Lamfalussy per atto (media articoli) ─────────────────────
LAMF_DIST = {}
if os.path.exists(NODES_LAMFALUSSY_FILE):
    lamf_df  = pd.read_csv(NODES_LAMFALUSSY_FILE)
    lamf_cols = [c for c in lamf_df.columns if c.startswith('lamf_L')]
    lamf_ok  = lamf_df[lamf_df['llm_status'] == 'ok']
    celex_col = 'celex' if 'celex' in lamf_ok.columns else 'id'
    if lamf_cols and not lamf_ok.empty:
        grp = lamf_ok.groupby(celex_col)[lamf_cols].mean().round(1)
        for act_id, row in grp.iterrows():
            LAMF_DIST[str(act_id)] = {k.replace('lamf_', ''): float(v)
                                       for k, v in row.items()}
    print(f"LAMF_DIST: {len(LAMF_DIST)} atti")
else:
    print("WARN: nodes_lamfalussy.csv non trovato — lamf non aggiunto ai nodi")

# ── 4. H Lamfalussy per atto da nodes_hybridity.csv ──────────────────────────
HYB_MAP = {}
if os.path.exists(NODES_HYBRIDITY_FILE):
    hyb_df   = pd.read_csv(NODES_HYBRIDITY_FILE)
    hid_col  = 'celex' if 'celex' in hyb_df.columns else 'id'
    for _, row in hyb_df.iterrows():
        HYB_MAP[str(row[hid_col])] = round(float(row.get('hybridity_score', 0)), 3)
    print(f"HYB_MAP:   {len(HYB_MAP)} atti")

# ── 5. Patch HTML ─────────────────────────────────────────────────────────────
with open(HTML_FILE, 'r', encoding='utf-8') as f:
    html = f.read()

# 5a. HEATMAPS (sostituisce HMxxx)
hm_js = 'const HEATMAPS = ' + _json.dumps(HEATMAPS, ensure_ascii=False) + ';'
html, n = re.subn(r'const HM\w+\s*=\s*\[[\s\S]*?\];', hm_js, html)
print(f"\nHM  sostituito: {n} occorrenza/e")

# 5b. L1 / L2
html, n = re.subn(r'const L1\s*=\s*\[.*?\];',
                   'const L1     = ' + _json.dumps(l1_arr) + ';', html)
print(f"L1  sostituito: {n}")
html, n = re.subn(r'const L2\s*=\s*\[.*?\];',
                   'const L2     = ' + _json.dumps(l2_arr) + ';', html)
print(f"L2  sostituito: {n}")

# 5c. NODES: heatmap, lamf, H
def patch_nodes(html, heatmap_ids, lamf_dist, hyb_map):
    m = re.search(r'(const NODES\s*=\s*)(\[[\s\S]*?\]);', html)
    if not m:
        print("WARN: const NODES non trovato")
        return html
    nodes = _json.loads(m.group(2))
    for nd in nodes:
        nid = nd['id']
        nd['heatmap'] = 'computed' if nid in heatmap_ids else nd.pop('heatmap', None) or None
        if nd.get('heatmap') is None:
            nd.pop('heatmap', None)
        if nid in lamf_dist:
            nd['lamf'] = lamf_dist[nid]
        else:
            nd.pop('lamf', None)
        if nid in hyb_map:
            nd['H'] = hyb_map[nid]
    new_block = m.group(1) + _json.dumps(nodes, ensure_ascii=False) + ';'
    lamf_n = sum(1 for nd in nodes if 'lamf' in nd)
    hm_n   = sum(1 for nd in nodes if nd.get('heatmap') == 'computed')
    print(f"NODES: {hm_n} con heatmap  |  {lamf_n} con lamf")
    return html[:m.start()] + new_block + html[m.end():]

html = patch_nodes(html, set(HEATMAPS.keys()), LAMF_DIST, HYB_MAP)

with open(HTML_FILE, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"\n✓ {HTML_FILE}")

Layer (5):
   0  'Scope-setting and' / 'framework clauses'
   1  'Procedural implementing' / 'provisions'
   2  'Technical implementing' / 'provisions'
   3  'Procedural derogation' / 'clauses'
   4  'Operational procedural' / 'micro-rules'

HEATMAPS: 19 atti  |  848 articoli totali
LAMF_DIST: 19 atti
HYB_MAP:   19 atti

HM  sostituito: 0 occorrenza/e
L1  sostituito: 1
L2  sostituito: 1
NODES: 19 con heatmap  |  19 con lamf

✓ ..\fdi_network.html
